<a href="https://colab.research.google.com/github/akriti404/Deep-Learning-Lab/blob/main/lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets



import torch

import torch.nn as nn

import torch.optim as optim

from torch.utils.data import DataLoader, Dataset

from datasets import load_dataset

from collections import Counter



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {device}")



print("Loading IMDb dataset...")



raw_data = load_dataset("stanfordnlp/imdb")



# Select 2,000 samples

train_subset = raw_data['train'].shuffle(seed=42).select(range(2000))

test_subset = raw_data['test'].shuffle(seed=42).select(range(500))



# Build Vocabulary from Training Set

word_counts = Counter()

for sample in train_subset:

    words = sample['text'].lower().split()

    word_counts.update(words)



# Keep top 10,000 most frequent words

MAX_VOCAB_SIZE = 10000

most_common_words = word_counts.most_common(MAX_VOCAB_SIZE - 2)



vocab = {'<PAD>': 0, '<UNK>': 1}

for word, _ in most_common_words:

    vocab[word] = len(vocab)



vocab_size = len(vocab)

MAX_LEN = 100  # Truncate/Pad sequences to 100 words



def text_to_tensor(text):

    tokens = [vocab.get(w.lower(), 1) for w in text.split()]

    if len(tokens) < MAX_LEN:

        tokens += [0] * (MAX_LEN - len(tokens))  # Padding

    else:

        tokens = tokens[:MAX_LEN]                 # Truncation

    return torch.tensor(tokens, dtype=torch.long)



class IMDbPyTorchDataset(Dataset):

    def __init__(self, hf_dataset):

        self.x = [text_to_tensor(sample['text']) for sample in hf_dataset]

        self.y = [torch.tensor(sample['label'], dtype=torch.float32) for sample in hf_dataset]



    def __len__(self):

        return len(self.x)



    def __getitem__(self, idx):

        return self.x[idx], self.y[idx]



train_dataset = IMDbPyTorchDataset(train_subset)

test_dataset = IMDbPyTorchDataset(test_subset)



train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)





# Standard Unidirectional Vanilla RNN

class VanillaRNN(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim):

        super(VanillaRNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, bidirectional=False)

        self.fc = nn.Linear(hidden_dim, 1)



    def forward(self, x):

        embedded = self.embedding(x)             # (batch_size, seq_len, embed_dim)

        out, h_n = self.rnn(embedded)

        last_hidden = h_n.squeeze(0)             # (batch_size, hidden_dim)

        logits = self.fc(last_hidden)            # (batch_size, 1)

        return logits.squeeze(1)



# Bidirectional RNN (BiRNN)

class BiRNN(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim):

        super(BiRNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, bidirectional=True)

        # Input features for FC layer are doubled due to [forward_h; backward_h]

        self.fc = nn.Linear(hidden_dim * 2, 1)



    def forward(self, x):

        embedded = self.embedding(x)             # (batch_size, seq_len, embed_dim)

        out, h_n = self.rnn(embedded)            # h_n shape: (2, batch_size, hidden_dim)



        # Extract and concatenate final forward and backward hidden states

        forward_hidden = h_n[-2, :, :]

        backward_hidden = h_n[-1, :, :]

        bidirectional_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)  # (batch_size, hidden_dim * 2)



        logits = self.fc(bidirectional_hidden)

        return logits.squeeze(1)

EMBED_DIM = 64

HIDDEN_DIM = 64



uni_model = VanillaRNN(vocab_size, EMBED_DIM, HIDDEN_DIM).to(device)

bi_model = BiRNN(vocab_size, EMBED_DIM, HIDDEN_DIM).to(device)



print("\n--- ARCHITECTURE SUMMARY ---")

print(f"Unidirectional FC Features: {uni_model.fc.in_features}")

print(f"Bidirectional FC Features:   {bi_model.fc.in_features}")





def train_and_evaluate(model, model_name, epochs=10):

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    criterion = nn.BCEWithLogitsLoss()



    print(f"\n==========================================")

    print(f"  Training {model_name}")

    print(f"==========================================")



    for epoch in range(1, epochs + 1):

        model.train()

        train_loss, train_correct, train_total = 0, 0, 0

        for bx, by in train_loader:

            bx, by = bx.to(device), by.to(device)

            optimizer.zero_grad()

            preds = model(bx)

            loss = criterion(preds, by)

            loss.backward()

            optimizer.step()



            train_loss += loss.item()

            probs = torch.sigmoid(preds)

            train_correct += ((probs >= 0.5).float() == by).sum().item()

            train_total += by.size(0)



        model.eval()

        test_loss, test_correct, test_total = 0, 0, 0

        with torch.no_grad():

            for bx, by in test_loader:

                bx, by = bx.to(device), by.to(device)

                preds = model(bx)

                loss = criterion(preds, by)



                test_loss += loss.item()

                probs = torch.sigmoid(preds)

                test_correct += ((probs >= 0.5).float() == by).sum().item()

                test_total += by.size(0)



        train_acc = (train_correct / train_total) * 100

        test_acc = (test_correct / test_total) * 100

        print(f"Epoch {epoch:02d} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")



#training

train_and_evaluate(uni_model, "Unidirectional RNN", epochs=10)

train_and_evaluate(bi_model, "Bidirectional RNN", epochs=10)



#test /inference

samples = [

    "The acting was not good at all, completely ruined the film.",

    "Not a bad movie, in fact it was remarkably entertaining.",

    "I expected it to be awful, but it turned out to be a brilliant masterpiece."

]



print("\n==========================================")

print("  INFERENCE COMPARISON ON TEST SENTENCES")

print("==========================================")



uni_model.eval()

bi_model.eval()



for sample in samples:

    input_tensor = text_to_tensor(sample).unsqueeze(0).to(device)

    with torch.no_grad():

        uni_prob = torch.sigmoid(uni_model(input_tensor)).item() * 100

        bi_prob = torch.sigmoid(bi_model(input_tensor)).item() * 100



    print(f"\nSentence: '{sample}'")

    print(f"  -> Unidirectional RNN Positivity Score: {uni_prob:.2f}%")

    print(f"  -> Bidirectional RNN Positivity Score:   {bi_prob:.2f}%")

Using device: cuda
Loading IMDb dataset...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]


--- ARCHITECTURE SUMMARY ---
Unidirectional FC Features: 64
Bidirectional FC Features:   128

  Training Unidirectional RNN
Epoch 01 | Train Acc: 50.25% | Test Acc: 52.60%
Epoch 02 | Train Acc: 56.15% | Test Acc: 53.60%
Epoch 03 | Train Acc: 59.75% | Test Acc: 52.60%
Epoch 04 | Train Acc: 64.85% | Test Acc: 52.20%
Epoch 05 | Train Acc: 69.90% | Test Acc: 54.20%
Epoch 06 | Train Acc: 73.15% | Test Acc: 52.40%
Epoch 07 | Train Acc: 78.25% | Test Acc: 52.80%
Epoch 08 | Train Acc: 81.05% | Test Acc: 53.80%
Epoch 09 | Train Acc: 84.35% | Test Acc: 53.60%
Epoch 10 | Train Acc: 87.90% | Test Acc: 52.60%

  Training Bidirectional RNN
Epoch 01 | Train Acc: 51.15% | Test Acc: 54.40%
Epoch 02 | Train Acc: 61.90% | Test Acc: 53.60%
Epoch 03 | Train Acc: 65.95% | Test Acc: 56.80%
Epoch 04 | Train Acc: 70.30% | Test Acc: 57.00%
Epoch 05 | Train Acc: 76.40% | Test Acc: 57.00%
Epoch 06 | Train Acc: 82.30% | Test Acc: 56.40%
Epoch 07 | Train Acc: 85.70% | Test Acc: 57.60%
Epoch 08 | Train Acc: 92.15% 